### Imports

In [ ]:
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import pytorch_lightning as pl

from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset, Sampler

from torchvision import models, datasets, transforms
import torchvision.transforms as T
from torchvision.datasets import ImageFolder

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

### Dataset

In [ ]:
# Dataset
pat = "2_png_224"

# All weights and saved models go here
WEIGHTS_DIR = "Transfer_learning/weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Tables
TABLE_PATH  = "ALL_MorphoSPLUS_GalfitM_output_splus.csv"
LABELS_PATH = "tabela_filtrada.csv"

# EfficientNet-B0: full fine-tuning (no freezing)

In [ ]:
# Load EfficientNet-B0 pretrained on ImageNet
# Full fine-tuning: all parameters trainable

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace classifier with a binary head (1 logit)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 1)
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")
print(f"Trainable %:      {100 * trainable / total:.4f}%")

## Augmentation and Dataloader

In [ ]:
# BalancedBatchSampler: always 50/50 batches
class BalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.targets    = torch.tensor(dataset.targets)
        assert batch_size % 2 == 0, "batch_size must be even"
        self.class0_idx     = torch.where(self.targets == 0)[0].tolist()
        self.class1_idx     = torch.where(self.targets == 1)[0].tolist()
        self.min_class_size = min(len(self.class0_idx), len(self.class1_idx))
        self.num_batches    = self.min_class_size // (batch_size // 2)

    def __iter__(self):
        random.shuffle(self.class0_idx)
        random.shuffle(self.class1_idx)
        for i in range(self.num_batches):
            batch0 = self.class0_idx[i*(self.batch_size//2):(i+1)*(self.batch_size//2)]
            batch1 = self.class1_idx[i*(self.batch_size//2):(i+1)*(self.batch_size//2)]
            batch  = batch0 + batch1
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches


# ImageNet normalization for EfficientNet
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.RandomRotation((0,   0)),
        transforms.RandomRotation((90,  90)),
        transforms.RandomRotation((180, 180)),
        transforms.RandomRotation((270, 270)),
    ]),
    transforms.RandomChoice([
        transforms.Compose([]),
        transforms.Compose([transforms.CenterCrop(179), transforms.Resize(224)]),
        transforms.Compose([transforms.CenterCrop(150), transforms.Resize(224)]),
        transforms.Compose([transforms.CenterCrop(112), transforms.Resize(224)]),
    ]),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_dataset = datasets.ImageFolder(root=f"{pat}/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=f"{pat}/val",   transform=eval_transforms)
test_dataset  = datasets.ImageFolder(root=f"{pat}/test",  transform=eval_transforms)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_sampler=BalancedBatchSampler(train_dataset, batch_size), num_workers=20)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=20)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=20)

print("Classes:", train_dataset.classes)
print("Train:",   len(train_dataset))
print("Val:",     len(val_dataset))
print("Test:",    len(test_dataset))

In [ ]:
# LightningModule: BCE + Binary Accuracy
from torchmetrics.classification import BinaryAccuracy

class LitEffNet(pl.LightningModule):
    def __init__(self, model, lr=3e-5, weight_decay=0.01):
        super().__init__()
        self.model = model
        self.save_hyperparameters(ignore=["model"])
        self.criterion = nn.BCEWithLogitsLoss()
        self.train_acc = BinaryAccuracy()
        self.val_acc   = BinaryAccuracy()

    def forward(self, x):
        return self.model(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss   = self.criterion(logits, labels.float())
        self.train_acc.update(torch.sigmoid(logits), labels)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def on_train_epoch_end(self):
        self.log("train_acc", self.train_acc.compute(), prog_bar=True)
        self.train_acc.reset()

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss   = self.criterion(logits, labels.float())
        self.val_acc.update(torch.sigmoid(logits), labels)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

    def on_validation_epoch_end(self):
        self.log("val_acc", self.val_acc.compute(), prog_bar=True)
        self.val_acc.reset()

    def configure_optimizers(self):
        return AdamW(filter(lambda p: p.requires_grad, self.parameters()),
                     lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)

In [ ]:
# Training
lit_effnet = LitEffNet(model)

ckpt_effnet = ModelCheckpoint(
    dirpath=WEIGHTS_DIR,
    filename="effnet-{epoch:03d}-{val_acc:.4f}",
    monitor="val_acc",
    mode="max",
    save_top_k=1
)

early_stop = EarlyStopping(
    monitor="val_acc",
    mode="max",
    patience=15,
    min_delta=1e-4,
    verbose=True
)

trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=10,
    logger=pl.loggers.TensorBoardLogger("tb_logs", name="efficientnet"),
    callbacks=[ckpt_effnet, early_stop]
)

trainer.fit(lit_effnet, train_loader, val_loader)
print("Best EfficientNet checkpoint:", ckpt_effnet.best_model_path)

---
# Inference & Ensemble
> Standalone section — loads the trained checkpoint and builds the tabular + EfficientNet stacking ensemble.

In [ ]:
import os, re, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import joblib
import pytorch_lightning as pl
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import models, datasets, transforms
from torchvision.datasets import ImageFolder
from torchmetrics.classification import BinaryAccuracy
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from imblearn.ensemble import BalancedRandomForestClassifier

SEED        = 42
pat         = "png_224"
WEIGHTS_DIR = "Transfer_learning/weights"
TABLE_PATH  = "ALL_MorphoSPLUS_GalfitM_output_splus.csv"
LABELS_PATH = "tabela_filtrada.csv"

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

pl.seed_everything(SEED)

In [ ]:
# LitEffNet definition (required to load checkpoint)
class LitEffNet(pl.LightningModule):
    def __init__(self, model, lr=3e-5, weight_decay=0.01):
        super().__init__()
        self.model = model
        self.save_hyperparameters(ignore=["model"])
        self.criterion = nn.BCEWithLogitsLoss()
        self.train_acc = BinaryAccuracy()
        self.val_acc   = BinaryAccuracy()

    def forward(self, x):
        return self.model(x).squeeze(1)

    def configure_optimizers(self):
        return AdamW(filter(lambda p: p.requires_grad, self.parameters()),
                     lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)

# Load best checkpoint
ckpt_files = glob.glob(f"{WEIGHTS_DIR}/effnet-*.ckpt")
best_ckpt  = max(ckpt_files, key=lambda p: float(p.split("val_acc=")[-1].replace(".ckpt", "")))
print("Loading:", best_ckpt)

base_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 1)
)

lit_effnet_inf = LitEffNet.load_from_checkpoint(best_ckpt, model=base_model)
lit_effnet_inf.eval()
print("Model loaded. Device:", next(lit_effnet_inf.parameters()).device)

In [ ]:
# Transforms + Loaders
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return img, label, self.samples[index][0]

def make_loader(root, batch_size=32, num_workers=4):
    ds = ImageFolderWithPaths(root=root, transform=eval_transforms)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

OID_RE = re.compile(r"(DR\d+_\d+_STRIPE82-\d{4}_\d{7})")

def object_id_from_path(p):
    m = OID_RE.search(os.path.basename(p))
    return m.group(1) if m else re.sub(r"\.[^.]+$", "", os.path.basename(p)).strip()

train_loader_inf = make_loader(f"{pat}/train")
val_loader_inf   = make_loader(f"{pat}/val")
test_loader_inf  = make_loader(f"{pat}/test")
print("Train:", len(train_loader_inf.dataset),
      "| Val:", len(val_loader_inf.dataset),
      "| Test:", len(test_loader_inf.dataset))

In [ ]:
# EfficientNet standalone evaluation
def plot_cm(cm, title):
    row_sums = cm.sum(axis=1, keepdims=True).astype(float); row_sums[row_sums==0] = 1
    cm_pct = cm / row_sums
    ann = [[f"{cm[i,j]}\n({cm_pct[i,j]*100:.1f}%)" for j in range(2)] for i in range(2)]
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_pct, annot=ann, fmt="", cmap="Blues", cbar=True,
                xticklabels=["Pred Good", "Pred Bad"],
                yticklabels=["True Good", "True Bad"])
    plt.title(title); plt.tight_layout(); plt.show()

def predict_probs(lit_model, dl, prob_col="effnet_prob"):
    lit_model.eval()
    device = next(lit_model.parameters()).device
    rows = []
    with torch.inference_mode():
        for images, labels, paths in dl:
            images = images.to(device, non_blocking=True)
            probs  = torch.sigmoid(lit_model(images)).cpu().numpy()
            for prob, label, path in zip(probs, labels.numpy(), paths):
                rows.append({"object_id": object_id_from_path(str(path)),
                             "y_true": int(label), prob_col: float(prob)})
    df = pd.DataFrame(rows)
    return df.groupby("object_id", as_index=False).agg({prob_col: "mean", "y_true": "first"})

df_effnet_val  = predict_probs(lit_effnet_inf, val_loader_inf)
df_effnet_test = predict_probs(lit_effnet_inf, test_loader_inf)

for split, df_e in [("Val", df_effnet_val), ("Test", df_effnet_test)]:
    y_true = df_e["y_true"].values
    pred   = (df_e["effnet_prob"].values >= 0.5).astype(int)
    print(f"\n── EfficientNet only [{split}] ──")
    print(classification_report(y_true, pred, target_names=["Good", "Bad"], zero_division=0))
    plot_cm(confusion_matrix(y_true, pred), f"EfficientNet only [{split}]")

In [ ]:
# Tabular data + feature selection
df_tab    = pd.read_csv(TABLE_PATH, low_memory=False)
df_labels = pd.read_csv(LABELS_PATH, low_memory=False)

df_tab["ID_1"]     = df_tab["ID_1"].astype(str).str.strip()
df_labels["ID"]    = df_labels["ID"].astype(str).str.strip()
df_labels["label"] = df_labels["type"].apply(lambda x: 0 if int(x) == 0 else 1)

text_cols = ["source_folder", "source_file", "ID_1", "Field_ID", "Dir_", "Field"]
for col in df_tab.columns:
    if col in text_cols: continue
    s = df_tab[col].astype(str).str.replace("*", "", regex=False).str.replace(",", ".", regex=False).str.strip()
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    df_tab[col] = pd.to_numeric(s, errors="coerce")

df_table = df_tab.merge(df_labels[["ID", "label"]], left_on="ID_1", right_on="ID", how="inner")
df_table["object_id"] = df_table["ID_1"].astype(str).str.strip()

def ids_from_loader(dl):
    ids = set()
    for _, _, paths in dl:
        ids.update(object_id_from_path(str(p)) for p in paths)
    return ids

train_ids = ids_from_loader(train_loader_inf)
val_ids   = ids_from_loader(val_loader_inf)
test_ids  = ids_from_loader(test_loader_inf)

df_train = df_table[df_table["object_id"].isin(train_ids)].copy()
df_val   = df_table[df_table["object_id"].isin(val_ids)].copy()
df_test  = df_table[df_table["object_id"].isin(test_ids)].copy()

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values

numeric_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != "label"]
features_all = [c for c in numeric_cols if df_train[c].notna().sum() >= 30 and df_train[c].nunique() >= 2]

resultados = []
for col in features_all:
    mask = df_train[col].notna().values
    if mask.sum() < 200: continue
    x_sub, y_sub = df_train.loc[mask, col].values, y_train[mask]
    if len(np.unique(y_sub)) < 2: continue
    try:
        auc = roc_auc_score(y_sub, x_sub)
        resultados.append([col, max(auc, 1 - auc)])
    except: pass

df_auc   = pd.DataFrame(resultados, columns=["feature", "auc_sep"]).sort_values("auc_sep", ascending=False)
auc_dict = dict(zip(df_auc["feature"], df_auc["auc_sep"]))
features_boas = df_auc[df_auc["auc_sep"] > 0.60]["feature"].tolist()

remover  = set()
corr_abs = df_train[features_boas].corr(method="pearson", min_periods=200).abs()
for i in range(len(features_boas)):
    for j in range(i + 1, len(features_boas)):
        f1, f2 = features_boas[i], features_boas[j]
        if f1 in remover or f2 in remover: continue
        if corr_abs.loc[f1, f2] >= 0.90:
            remover.add(f2 if auc_dict[f1] >= auc_dict[f2] else f1)

features_sel = [f for f in features_boas if f not in remover]
print(f"Selected tabular features: {len(features_sel)}")
print(features_sel)

In [ ]:
# Balanced Random Forest (tabular)
def make_rf_pipeline(seed=SEED):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", BalancedRandomForestClassifier(
            n_estimators=1000, random_state=seed, n_jobs=-1, min_samples_leaf=2
        ))
    ])

rf_tab = make_rf_pipeline()
rf_tab.fit(df_train[features_sel], y_train)

for split, X, y in [("Val", df_val[features_sel], y_val), ("Test", df_test[features_sel], y_test)]:
    prob = rf_tab.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print(f"\n── BRF tabular [{split}] ──")
    print(classification_report(y, pred, target_names=["Good", "Bad"], zero_division=0))
    plot_cm(confusion_matrix(y, pred), f"BRF tabular [{split}]")

In [ ]:
# Stacking Ensemble: BRF + EfficientNet
# Meta-model trained on VAL, evaluated on VAL and TEST

# df_effnet_val / df_effnet_test already computed above
df_effnet_train = predict_probs(lit_effnet_inf, train_loader_inf)

def get_effnet_probs(df_split, df_effnet):
    return df_split[["object_id"]].merge(df_effnet, on="object_id", how="left")["effnet_prob"].fillna(0.5).values

effnet_val  = get_effnet_probs(df_val,  df_effnet_val)
effnet_test = get_effnet_probs(df_test, df_effnet_test)

brf_val  = rf_tab.predict_proba(df_val[features_sel])[:, 1]
brf_test = rf_tab.predict_proba(df_test[features_sel])[:, 1]

X_meta_val  = np.column_stack([brf_val,  effnet_val])
X_meta_test = np.column_stack([brf_test, effnet_test])

meta = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
meta.fit(X_meta_val, y_val)

print("Stacking weights:")
for name, w in zip(["BRF", "EfficientNet"], meta.coef_[0]):
    print(f"  {name}: {w:.4f}")

for split, X_meta, y_true in [("Val", X_meta_val, y_val), ("Test", X_meta_test, y_test)]:
    prob = meta.predict_proba(X_meta)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print(f"\n── Stacking BRF + EfficientNet [{split}] ──")
    print(classification_report(y_true, pred, target_names=["Good", "Bad"], zero_division=0))
    plot_cm(confusion_matrix(y_true, pred), f"Stacking BRF + EfficientNet [{split}]")

In [ ]:
# Save all trained models to weights dir
import joblib

joblib.dump(rf_tab,       f"{WEIGHTS_DIR}/brf_tabular.pkl")
joblib.dump(meta,         f"{WEIGHTS_DIR}/meta_lr_effnet.pkl")
joblib.dump(features_sel, f"{WEIGHTS_DIR}/features_sel.pkl")

print("Saved to:", WEIGHTS_DIR)
print("  brf_tabular.pkl")
print("  meta_lr_effnet.pkl")
print("  features_sel.pkl")
print("  effnet-*.ckpt  (saved during training by ModelCheckpoint)")

---
# Prediction on New Dataset
> Standalone section — loads saved BRF, EfficientNet and meta-model to run predictions on new images and table. Labels are not required.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torchmetrics.classification import BinaryAccuracy
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix

# Paths - adjust for your new dataset
NEW_IMG_DIR  = "png_224_new"                              # image folder (needs at least one subfolder)
NEW_TABLE    = "nuevo_dataset.csv"                        # morphological features table
NEW_ID_COL   = "ID_1"                                     # ID column in the new table
OUTPUT_CSV   = "Transfer_learning/predictions_new_effnet.csv"
WEIGHTS_DIR  = "Transfer_learning/weights"
SEED         = 42

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

pl.seed_everything(SEED)

In [ ]:
# Load EfficientNet
class LitEffNet(pl.LightningModule):
    def __init__(self, model, lr=3e-5, weight_decay=0.01):
        super().__init__()
        self.model = model
        self.save_hyperparameters(ignore=["model"])
        self.criterion = nn.BCEWithLogitsLoss()
        self.train_acc = BinaryAccuracy()
        self.val_acc   = BinaryAccuracy()

    def forward(self, x):
        return self.model(x).squeeze(1)

    def configure_optimizers(self):
        return AdamW(filter(lambda p: p.requires_grad, self.parameters()),
                     lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)

ckpt_files = glob.glob(f"{WEIGHTS_DIR}/effnet-*.ckpt")
best_ckpt  = max(ckpt_files, key=lambda p: float(p.split("val_acc=")[-1].replace(".ckpt", "")))
print("EfficientNet checkpoint:", best_ckpt)

base_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 1)
)

lit_effnet = LitEffNet.load_from_checkpoint(best_ckpt, model=base_model)
lit_effnet.eval()
device = next(lit_effnet.parameters()).device
print("EfficientNet loaded. Device:", device)

# Load BRF + meta-model
rf_tab       = joblib.load(f"{WEIGHTS_DIR}/brf_tabular.pkl")
meta         = joblib.load(f"{WEIGHTS_DIR}/meta_lr_effnet.pkl")
features_sel = joblib.load(f"{WEIGHTS_DIR}/features_sel.pkl")
print(f"BRF and meta loaded. Features: {len(features_sel)}")

In [ ]:
# New image loader + EfficientNet inference
# If images have no labels, put them all in: NEW_IMG_DIR/unknown/

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

OID_RE = re.compile(r"(DR\d+_\d+_STRIPE82-\d{4}_\d{7})")

def object_id_from_path(p):
    m = OID_RE.search(os.path.basename(p))
    return m.group(1) if m else re.sub(r"\.[^.]+$", "", os.path.basename(p)).strip()

class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return img, label, self.samples[index][0]

new_ds     = ImageFolderWithPaths(root=NEW_IMG_DIR, transform=eval_transforms)
new_loader = DataLoader(new_ds, batch_size=32, shuffle=False, num_workers=4)
print(f"New images: {len(new_ds)}")

rows = []
with torch.inference_mode():
    for images, _, paths in new_loader:
        images = images.to(device, non_blocking=True)
        probs  = torch.sigmoid(lit_effnet(images)).cpu().numpy()
        for prob, path in zip(probs, paths):
            rows.append({"object_id": object_id_from_path(str(path)), "effnet_prob": float(prob)})

df_effnet_new = pd.DataFrame(rows).groupby("object_id", as_index=False).agg({"effnet_prob": "mean"})
print(f"Unique objects (images): {len(df_effnet_new)}")

In [ ]:
# Load tabular + BRF inference + ensemble prediction
df_new_tab = pd.read_csv(NEW_TABLE, low_memory=False)
df_new_tab[NEW_ID_COL] = df_new_tab[NEW_ID_COL].astype(str).str.strip()

text_cols = ["source_folder", "source_file", "ID_1", "Field_ID", "Dir_", "Field"]
for col in df_new_tab.columns:
    if col in text_cols: continue
    s = df_new_tab[col].astype(str).str.replace("*", "", regex=False).str.replace(",", ".", regex=False).str.strip()
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    df_new_tab[col] = pd.to_numeric(s, errors="coerce")

df_new_tab["object_id"] = df_new_tab[NEW_ID_COL]

missing_feats = [f for f in features_sel if f not in df_new_tab.columns]
if missing_feats:
    print(f"Warning: {len(missing_feats)} features missing, will be imputed as NaN:")
    print(missing_feats)
    for f in missing_feats:
        df_new_tab[f] = np.nan

brf_prob_new = rf_tab.predict_proba(df_new_tab[features_sel])[:, 1]

df_result = df_new_tab[["object_id"]].copy()
df_result["brf_prob"] = brf_prob_new
df_result = df_result.merge(df_effnet_new, on="object_id", how="left")
df_result["effnet_prob"] = df_result["effnet_prob"].fillna(0.5)

X_new = np.column_stack([df_result["brf_prob"].values, df_result["effnet_prob"].values])
df_result["ensemble_prob"] = meta.predict_proba(X_new)[:, 1]
df_result["pred_label"]    = (df_result["ensemble_prob"] >= 0.5).astype(int)
df_result["pred_class"]    = df_result["pred_label"].map({0: "Good", 1: "Bad"})

print(df_result[["object_id", "brf_prob", "effnet_prob", "ensemble_prob", "pred_class"]].head(10))
print(f"\nTotal: {len(df_result)} | Good: {(df_result.pred_label==0).sum()} | Bad: {(df_result.pred_label==1).sum()}")

In [ ]:
# Save predictions
df_result.to_csv(OUTPUT_CSV, index=False)
print(f"Predictions saved to: {OUTPUT_CSV}")

# If the new dataset has labels, uncomment to evaluate:
# LABEL_COL = "type"   # column with 0=Good / 1=Bad
# if LABEL_COL in df_new_tab.columns:
#     y_true = df_new_tab[LABEL_COL].values
#     print(classification_report(y_true, df_result["pred_label"].values,
#                                 target_names=["Good", "Bad"], zero_division=0))
#     cm = confusion_matrix(y_true, df_result["pred_label"].values)
#     row_sums = cm.sum(axis=1, keepdims=True).astype(float); row_sums[row_sums==0] = 1
#     cm_pct = cm / row_sums
#     ann = [[f'{cm[i,j]}\n({cm_pct[i,j]*100:.1f}%)' for j in range(2)] for i in range(2)]
#     plt.figure(figsize=(5,4))
#     sns.heatmap(cm_pct, annot=ann, fmt="", cmap="Blues", cbar=True,
#                 xticklabels=["Pred Good","Pred Bad"], yticklabels=["True Good","True Bad"])
#     plt.title("Ensemble - New dataset"); plt.tight_layout(); plt.show()